## Neural Network Project Part 1: Image Classification




In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Part 1: Image classification
## 1.1 Warming up:
### 1. Train classifier on the CIFAR 100 dataset

In [ ]:
# Frirstly we start by importing the necessary libraries needed
import tensorflow as tf
print(tf.__version__)
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout
from sklearn.preprocessing import OneHotEncoder



2.15.0


###Loading of the CIFAR 100 dataset and scaling them in the range of 0-255






In [ ]:
# # Frirstly we start by loading the CIFAR 100 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()
# Load CIFAR 100 dataset with fine-grained labels
#(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data(label_mode='fine')

# Normalize pixel values to be between 0 and 1
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Print the shape of the loaded data
print("Training data shape:", x_train.shape)
print("Training labels shape:", y_train.shape)
print("Test data shape:", x_test.shape)
print("Test labels shape:", y_test.shape)

169001437/169001437 [==============================] - 3s 0us/step
Training data shape: (50000, 32, 32, 3)
Training labels shape: (50000, 1)
Test data shape: (10000, 32, 32, 3)
Test labels shape: (10000, 1)


### Prepocessing the data
Firstly, the images are preprocessed using
tf.keras.applications.vgg16.preprocess_input for vgg16

In [ ]:
# Preprocess the training and test data
x_train_preprocessed = preprocess_input(x_train)
x_test_preprocessed = preprocess_input(x_test)

# Print the shape of preprocessed data
print("Preprocessed training data shape:", x_train_preprocessed.shape)
print("Preprocessed test data shape:", x_test_preprocessed.shape)

Preprocessed training data shape: (50000, 32, 32, 3)
Preprocessed test data shape: (10000, 32, 32, 3)


### Feature extraction and training the dense layer network
Here are the steps:
* The pre-trained VGG16 model is loaded without the top layers.
* The weights of the convolutional base are freezed to prevent them from being updated during training.
* Add Dense layers on top of the extracted features.
* Compile the model with an optimizer, loss function, and evaluation metric.
* Train the model on the preprocessed training data.
* Evaluate the model on the preprocessed test data.

From the result, a test accuracy of 36.95 % is obtained

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten
from sklearn.preprocessing import OneHotEncoder

# Load the CIFAR 100 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()

# Preprocess the images
x_train_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_train.astype('float32'))
x_test_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_test.astype('float32'))

# One-hot encode the labels
encoder = OneHotEncoder(sparse=False)
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

# Load the pre-trained VGG16 model without the top (fully connected) layers
vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Freeze the weights of the convolutional base
for layer in vgg_base.layers:
    layer.trainable = False

# Add Dense layers on top of the extracted features
x = Flatten()(vgg_base.output)
x = Dense(512, activation='relu')(x)
x = Dense(256, activation='relu')(x)
predictions = Dense(100, activation='softmax')(x)

# Create the final model
model = Model(inputs=vgg_base.input, outputs=predictions)

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(x_train_preprocessed, y_train_encoded, epochs=10, batch_size=128, validation_data=(x_test_preprocessed, y_test_encoded))

# Evaluate the model
test_loss, test_acc = model.evaluate(x_test_preprocessed, y_test_encoded)
print('Test accuracy:', test_acc)


/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


58889256/58889256 [==============================] - 0s 0us/step
Epoch 1/10
391/391 [==============================] - 14s 23ms/step - loss: 3.5977 - accuracy: 0.2306 - val_loss: 2.7868 - val_accuracy: 0.3185
Epoch 2/10
391/391 [==============================] - 7s 19ms/step - loss: 2.4250 - accuracy: 0.3810 - val_loss: 2.5775 - val_accuracy: 0.3586
Epoch 3/10
391/391 [==============================] - 7s 19ms/step - loss: 2.0655 - accuracy: 0.4518 - val_loss: 2.5450 - val_accuracy: 0.3769
Epoch 4/10
391/391 [==============================] - 7s 19ms/step - loss: 1.7935 - accuracy: 0.5101 - val_loss: 2.6008 - val_accuracy: 0.3790
Epoch 5/10
391/391 [==============================] - 8s 19ms/step - loss: 1.5740 - accuracy: 0.5591 - val_loss: 2.7103 - val_accuracy: 0.3718
Epoch 6/10
391/391 [==============================] - 7s 19ms/step - loss: 1.3532 - accuracy: 0.6133 - val_loss: 2.8416 - val_accuracy: 0.3722
Epoch 7/10
391/391 [==============================] - 7s 19ms/step - loss: 1

#### Alternating the model 1

Alternating the model by changing some parameters is done so as to get an improved accuracy.
In this configuration, more Dense layers are added with different activation functions (ReLU) and dropout rates (0.5, 0.3, and 0.2). Dropout is a regularization technique that randomly drops a fraction of the units (neurons) during training to prevent overfitting.

After running the code, we get an accuracy of about 36.28%, which shows little or no improvement form the first model. We will proceed with more  alternating and observe the results

In [ ]:

# Load the CIFAR 100 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()

# Preprocess the images
x_train_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_train.astype('float32'))
x_test_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_test.astype('float32'))

# One-hot encode the labels
encoder = OneHotEncoder(sparse=False)
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

# Load the pre-trained VGG16 model without the top (fully connected) layers
vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Freeze the weights of the convolutional base
for layer in vgg_base.layers:
    layer.trainable = False

# Add Dense layers on top of the extracted features
x = Flatten()(vgg_base.output)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
predictions = Dense(100, activation='softmax')(x)

# Create the final model
model = Model(inputs=vgg_base.input, outputs=predictions)

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(x_train_preprocessed, y_train_encoded, epochs=10, batch_size=128, validation_data=(x_test_preprocessed, y_test_encoded))

# Evaluate the model
test_loss, test_acc = model.evaluate(x_test_preprocessed, y_test_encoded)
print('Test accuracy:', test_acc)


/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Epoch 1/10
391/391 [==============================] - 11s 21ms/step - loss: 4.8769 - accuracy: 0.0373 - val_loss: 3.9145 - val_accuracy: 0.1247
Epoch 2/10
391/391 [==============================] - 8s 20ms/step - loss: 3.7416 - accuracy: 0.1226 - val_loss: 3.1397 - val_accuracy: 0.2286
Epoch 3/10
391/391 [==============================] - 8s 20ms/step - loss: 3.3119 - accuracy: 0.1851 - val_loss: 2.9056 - val_accuracy: 0.2755
Epoch 4/10
391/391 [==============================] - 8s 19ms/step - loss: 3.1105 - accuracy: 0.2215 - val_loss: 2.7965 - val_accuracy: 0.3044
Epoch 5/10
391/391 [==============================] - 8s 20ms/step - loss: 2.9971 - accuracy: 0.2432 - val_loss: 2.7343 - val_accuracy: 0.3181
Epoch 6/10
391/391 [==============================] - 8s 20ms/step - loss: 2.9008 - accuracy: 0.2635 - val_loss: 2.6643 - val_accuracy: 0.3279
Epoch 7/10
391/391 [==============================] - 8s 20ms/step - loss: 2.8250 - accuracy: 0.2785 - val_loss: 2.6357 - val_accuracy: 0.342

#### Alternating the model 2
In this code, the SGD optimizer is used with a learning rate of 0.001 and momentum of 0.9.
The model is trained with 15 epochs.

Here, we observe a slight improvement of 37.3 % accuracy

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import SGD
from sklearn.preprocessing import OneHotEncoder

# Load the CIFAR 100 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()

# Preprocess the images
x_train_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_train.astype('float32'))
x_test_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_test.astype('float32'))

# One-hot encode the labels
encoder = OneHotEncoder(sparse=False)
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

# Load the pre-trained VGG16 model without the top (fully connected) layers
vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Freeze the weights of the convolutional base
for layer in vgg_base.layers:
    layer.trainable = False


# Add Dense layers on top of the extracted features
x = Flatten()(vgg_base.output)
x = Dense(512, activation='relu')(x)
x = Dense(256, activation='relu')(x)
predictions = Dense(100, activation='softmax')(x)

# Create the final model
model = Model(inputs=vgg_base.input, outputs=predictions)

# Compile the model with SGD optimizer and different learning rate
sgd_optimizer = SGD(learning_rate=0.001, momentum=0.9)
model.compile(optimizer=sgd_optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model with different number of epochs
history = model.fit(x_train_preprocessed, y_train_encoded, epochs=15, batch_size=128, validation_data=(x_test_preprocessed, y_test_encoded))

# Evaluate the model
test_loss, test_acc = model.evaluate(x_test_preprocessed, y_test_encoded)
print('Test accuracy:', test_acc)


/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Epoch 1/15
391/391 [==============================] - 9s 21ms/step - loss: 4.2929 - accuracy: 0.1507 - val_loss: 3.4049 - val_accuracy: 0.2263
Epoch 2/15
391/391 [==============================] - 7s 18ms/step - loss: 3.0317 - accuracy: 0.2794 - val_loss: 3.0017 - val_accuracy: 0.2810
Epoch 3/15
391/391 [==============================] - 7s 19ms/step - loss: 2.6565 - accuracy: 0.3409 - val_loss: 2.8392 - val_accuracy: 0.3128
Epoch 4/15
391/391 [==============================] - 7s 19ms/step - loss: 2.4235 - accuracy: 0.3840 - val_loss: 2.7422 - val_accuracy: 0.3277
Epoch 5/15
391/391 [==============================] - 7s 19ms/step - loss: 2.2527 - accuracy: 0.4184 - val_loss: 2.6912 - val_accuracy: 0.3405
Epoch 6/15
391/391 [==============================] - 8s 20ms/step - loss: 2.1157 - accuracy: 0.4477 - val_loss: 2.6525 - val_accuracy: 0.3497
Epoch 7/15
391/391 [==============================] - 7s 19ms/step - loss: 1.9975 - accuracy: 0.4758 - val_loss: 2.6427 - val_accuracy: 0.3564

####Alternating 3 (best model)
In this code:  
* 3 dense layers and 5 epochs are used.
* Adam optimizer is used
* Activation function: ReLU

The accuracy obtained here is : 38.26%

This is the best model gotten, after experimenting with different configurations like the number of Dense layers, activation functions, optimizers, and epochs.
This may suggest that a simpler architecture with fewer layers and epochs is better for this task.

  

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import SGD
from sklearn.preprocessing import OneHotEncoder

# Load the CIFAR 100 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()

# Preprocess the images
x_train_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_train.astype('float32'))
x_test_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_test.astype('float32'))

# One-hot encode the labels
encoder = OneHotEncoder(sparse=False)
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

# Load the pre-trained VGG16 model without the top (fully connected) layers
vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Freeze the weights of the convolutional base
for layer in vgg_base.layers:
    layer.trainable = False


# Add Dense layers on top of the extracted features
x = Flatten()(vgg_base.output)
x = Dense(512, activation='relu')(x)
x = Dense(256, activation='relu')(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(100, activation='softmax')(x)

# Create the final model
model = Model(inputs=vgg_base.input, outputs=predictions)

# Compile the model with SGD optimizer and different learning rate
#sgd_optimizer = SGD(learning_rate=0.001, momentum=0.9)
#model.compile(optimizer=sgd_optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model with different number of epochs
history = model.fit(x_train_preprocessed, y_train_encoded, epochs=5, batch_size=128, validation_data=(x_test_preprocessed, y_test_encoded))

# Evaluate the model
test_loss, test_acc = model.evaluate(x_test_preprocessed, y_test_encoded)
print('Test accuracy:', test_acc)

/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Epoch 1/5
391/391 [==============================] - 10s 22ms/step - loss: 3.4542 - accuracy: 0.2170 - val_loss: 2.7603 - val_accuracy: 0.3141
Epoch 2/5
391/391 [==============================] - 8s 21ms/step - loss: 2.4589 - accuracy: 0.3680 - val_loss: 2.5726 - val_accuracy: 0.3513
Epoch 3/5
391/391 [==============================] - 8s 20ms/step - loss: 2.1399 - accuracy: 0.4359 - val_loss: 2.5234 - val_accuracy: 0.3676
Epoch 4/5
391/391 [==============================] - 8s 20ms/step - loss: 1.8933 - accuracy: 0.4865 - val_loss: 2.5226 - val_accuracy: 0.3827
Epoch 5/5
313/313 [==============================] - 3s 8ms/step - loss: 2.5865 - accuracy: 0.3837
Test accuracy: 0.38370001316070557


####Alternating 4
Another model modification is done to further try to improve the results obtained.
* 3 dense layers and an increase to 20 epochs

From the results,there is rather a drop in the accuracy to 36%

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import SGD
from sklearn.preprocessing import OneHotEncoder

# Load the CIFAR 100 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()

# Preprocess the images
x_train_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_train.astype('float32'))
x_test_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_test.astype('float32'))

# One-hot encode the labels
encoder = OneHotEncoder(sparse=False)
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

# Load the pre-trained VGG16 model without the top (fully connected) layers
vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Freeze the weights of the convolutional base
for layer in vgg_base.layers:
    layer.trainable = False


# Add Dense layers on top of the extracted features
x = Flatten()(vgg_base.output)
x = Dense(512, activation='relu')(x)
x = Dense(256, activation='relu')(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(100, activation='softmax')(x)

# Create the final model
model = Model(inputs=vgg_base.input, outputs=predictions)

# Compile the model with SGD optimizer and different learning rate
#sgd_optimizer = SGD(learning_rate=0.001, momentum=0.9)
#model.compile(optimizer=sgd_optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model with different number of epochs
history = model.fit(x_train_preprocessed, y_train_encoded, epochs=20, batch_size=128, validation_data=(x_test_preprocessed, y_test_encoded))

# Evaluate the model
test_loss, test_acc = model.evaluate(x_test_preprocessed, y_test_encoded)
print('Test accuracy:', test_acc)

/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Epoch 1/20
391/391 [==============================] - 10s 21ms/step - loss: 3.4104 - accuracy: 0.2259 - val_loss: 2.7517 - val_accuracy: 0.3205
Epoch 2/20
391/391 [==============================] - 8s 20ms/step - loss: 2.4460 - accuracy: 0.3731 - val_loss: 2.5791 - val_accuracy: 0.3541
Epoch 3/20
391/391 [==============================] - 8s 20ms/step - loss: 2.1268 - accuracy: 0.4388 - val_loss: 2.4941 - val_accuracy: 0.3722
Epoch 4/20
391/391 [==============================] - 8s 20ms/step - loss: 1.8781 - accuracy: 0.4929 - val_loss: 2.5383 - val_accuracy: 0.3777
Epoch 5/20
391/391 [==============================] - 8s 20ms/step - loss: 1.6635 - accuracy: 0.5391 - val_loss: 2.6498 - val_accuracy: 0.3802
Epoch 6/20
391/391 [==============================] - 8s 20ms/step - loss: 1.4519 - accuracy: 0.5924 - val_loss: 2.7568 - val_accuracy: 0.3746
Epoch 7/20
391/391 [==============================] - 8s 20ms/step - loss: 1.2646 - accuracy: 0.6359 - val_loss: 2.8429 - val_accuracy: 0.372

#### Alternating 5

With this modification, we are using 3 dense layers and 20 epochs and a batch size of 512.

We observe a slight improvement of 1% from the previous model, but which is still not the best. We therefore can conclude that the model with the accuracy of 38.2% is best.


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import SGD
from sklearn.preprocessing import OneHotEncoder

# Load the CIFAR 100 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()

# Preprocess the images
x_train_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_train.astype('float32'))
x_test_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_test.astype('float32'))

# One-hot encode the labels
encoder = OneHotEncoder(sparse=False)
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

# Load the pre-trained VGG16 model without the top (fully connected) layers
vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Freeze the weights of the convolutional base
for layer in vgg_base.layers:
    layer.trainable = False


# Add Dense layers on top of the extracted features
x = Flatten()(vgg_base.output)
x = Dense(512, activation='relu')(x)
x = Dense(256, activation='relu')(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(100, activation='softmax')(x)

# Create the final model
model = Model(inputs=vgg_base.input, outputs=predictions)

# Compile the model with SGD optimizer and different learning rate
#sgd_optimizer = SGD(learning_rate=0.001, momentum=0.9)
#model.compile(optimizer=sgd_optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model with different number of epochs
history = model.fit(x_train_preprocessed, y_train_encoded, epochs=10, batch_size = 512, validation_data=(x_test_preprocessed, y_test_encoded))

# Evaluate the model
test_loss, test_acc = model.evaluate(x_test_preprocessed, y_test_encoded)
print('Test accuracy:', test_acc)

/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Epoch 1/10
98/98 [==============================] - 20s 124ms/step - loss: 4.2459 - accuracy: 0.1339 - val_loss: 3.1689 - val_accuracy: 0.2613
Epoch 2/10
98/98 [==============================] - 6s 66ms/step - loss: 2.7678 - accuracy: 0.3212 - val_loss: 2.7139 - val_accuracy: 0.3330
Epoch 3/10
98/98 [==============================] - 6s 63ms/step - loss: 2.3309 - accuracy: 0.4035 - val_loss: 2.5940 - val_accuracy: 0.3574
Epoch 4/10
98/98 [==============================] - 6s 61ms/step - loss: 2.0546 - accuracy: 0.4584 - val_loss: 2.5555 - val_accuracy: 0.3677
Epoch 5/10
98/98 [==============================] - 6s 62ms/step - loss: 1.8344 - accuracy: 0.5097 - val_loss: 2.5677 - val_accuracy: 0.3744
Epoch 6/10
98/98 [==============================] - 6s 64ms/step - loss: 1.6340 - accuracy: 0.5565 - val_loss: 2.6114 - val_accuracy: 0.3789
Epoch 7/10
98/98 [==============================] - 6s 61ms/step - loss: 1.4393 - accuracy: 0.6036 - val_loss: 2.6987 - val_accuracy: 0.3777
Epoch 8/10


### 2. Fine-tune the upper block of the convolutional base of the VGG16 network.
In this code:

We load the CIFAR 100 dataset and preprocess the images, then we one-hot encode the labels.
Furthermore, the pre-trained VGG16 model with the convolutional base is loaded and all other layers are frozen. A dense layers is also added on top of the convolutional base for classification.

The model is compiled with RMSprop optimizer and a lower learning rate and the model is trained and evaluated on the test data.

This code will fine-tune the upper block of the VGG16 network and add a dense classification layer on top.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import RMSprop
from sklearn.preprocessing import OneHotEncoder

# Load the CIFAR 100 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()

# Preprocess the images
x_train_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_train.astype('float32'))
x_test_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_test.astype('float32'))

# One-hot encode the labels
encoder = OneHotEncoder(sparse=False)
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

# Load the pre-trained VGG16 model with the convolutional base
vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Freeze layers up to a certain point
for layer in vgg_base.layers[:-4]:
    layer.trainable = False

# Add Dense layers on top of the convolutional base
x = Flatten()(vgg_base.output)
x = Dense(512, activation='relu')(x)
x = Dense(256, activation='relu')(x)
predictions = Dense(100, activation='softmax')(x)

# Create the final model
model = Model(inputs=vgg_base.input, outputs=predictions)

# Compile the model with RMSprop optimizer and a lower learning rate
rmsprop_optimizer = RMSprop(learning_rate=0.0001)
model.compile(optimizer=rmsprop_optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Print a summary of the model
model.summary()

# Train the model
history = model.fit(x_train_preprocessed, y_train_encoded, epochs=10, batch_size=128, validation_data=(x_test_preprocessed, y_test_encoded))

# Evaluate the model
test_loss, test_acc = model.evaluate(x_test_preprocessed, y_test_encoded)
print('Test accuracy:', test_acc)


/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Model: "model_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_8 (InputLayer)        [(None, 32, 32, 3)]       0         
                                                                 
 block1_conv1 (Conv2D)       (None, 32, 32, 64)        1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 32, 32, 64)        36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 16, 16, 64)        0         
                                                                 
 block2_conv1 (Conv2D)       (None, 16, 16, 128)       73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 16, 16, 128)       147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 8, 8, 128)         0   

### Results from fine tunning the upperblock
After fine tunning, we observe an increase in the accuracy which is now 50%.
This is because:
* Transfer Learning: Leveraging pre-trained weights from VGG16 on the ImageNet dataset provides a strong starting point for learning features relevant to the CIFAR-100 dataset, leading to better performance.
* The upper layers of VGG16 capture high-level features that are more relevant to specific tasks. Fine-tuning these layers allows the model to adapt these features to better distinguish between CIFAR-100 classes.
* Increased Model Capacity: Fine-tuning increases the model's capacity to learn from CIFAR-100 by adjusting the weights of the upper layers, enabling better capture of dataset complexities.


In the next block, we will do data augmentation to the model and observe the results.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.preprocessing import OneHotEncoder

# Load the CIFAR 100 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()

# Preprocess the images
x_train_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_train.astype('float32'))
x_test_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_test.astype('float32'))

# One-hot encode the labels
encoder = OneHotEncoder(sparse=False)
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

# Load the pre-trained VGG16 model with the convolutional base
vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Freeze layers up to a certain point
for layer in vgg_base.layers[:-4]:
    layer.trainable = False

# Add Dense layers on top of the convolutional base
x = Flatten()(vgg_base.output)
x = Dense(512, activation='relu')(x)
x = Dense(256, activation='relu')(x)
predictions = Dense(100, activation='softmax')(x)

# Create the final model
model = Model(inputs=vgg_base.input, outputs=predictions)

# Compile the model with RMSprop optimizer and a lower learning rate
rmsprop_optimizer = RMSprop(learning_rate=0.0001)
model.compile(optimizer=rmsprop_optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Print a summary of the model
model.summary()

# Data augmentation
datagen = ImageDataGenerator(
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Train the model with data augmentation
history = model.fit(datagen.flow(x_train_preprocessed, y_train_encoded, batch_size=128),
                    steps_per_epoch=len(x_train_preprocessed) / 128,
                    epochs=10,
                    validation_data=(x_test_preprocessed, y_test_encoded))

# Evaluate the model
test_loss, test_acc = model.evaluate(x_test_preprocessed, y_test_encoded)
print('Test accuracy:', test_acc)


/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Model: "model_8"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_9 (InputLayer)        [(None, 32, 32, 3)]       0         
                                                                 
 block1_conv1 (Conv2D)       (None, 32, 32, 64)        1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 32, 32, 64)        36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 16, 16, 64)        0         
                                                                 
 block2_conv1 (Conv2D)       (None, 16, 16, 128)       73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 16, 16, 128)       147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 8, 8, 128)         0   

From the initial data augmentation step, the accuracy drops to 46.8%.

I will do some adjustments to  the parameters of the data augmentation inorder to get improved results.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.preprocessing import OneHotEncoder

# Load the CIFAR 100 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()

# Preprocess the images
x_train_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_train.astype('float32'))
x_test_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_test.astype('float32'))

# One-hot encode the labels
encoder = OneHotEncoder(sparse=False)
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

# Load the pre-trained VGG16 model with the convolutional base
vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Freeze layers up to a certain point
for layer in vgg_base.layers[:-4]:
    layer.trainable = False

# Add Dense layers on top of the convolutional base
x = Flatten()(vgg_base.output)
x = Dense(512, activation='relu')(x)
x = Dense(256, activation='relu')(x)
predictions = Dense(100, activation='softmax')(x)

# Create the final model
model = Model(inputs=vgg_base.input, outputs=predictions)

# Compile the model with RMSprop optimizer and a lower learning rate
rmsprop_optimizer = RMSprop(learning_rate=0.0001)
model.compile(optimizer=rmsprop_optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Print a summary of the model
model.summary()

# Data augmentation
datagen = ImageDataGenerator(
    rotation_range=20,             # Reduce rotation range
    width_shift_range=0.1,         # Reduce width shift range
    height_shift_range=0.1,        # Reduce height shift range
    shear_range=0.1,               # Reduce shear range
    zoom_range=0.1,                # Reduce zoom range
    horizontal_flip=True,
    fill_mode='nearest'
)

# Train the model with data augmentation
history = model.fit(datagen.flow(x_train_preprocessed, y_train_encoded, batch_size=128),
                    steps_per_epoch=len(x_train_preprocessed) / 128,
                    epochs=10,
                    validation_data=(x_test_preprocessed, y_test_encoded))

# Evaluate the model
test_loss, test_acc = model.evaluate(x_test_preprocessed, y_test_encoded)
print('Test accuracy:', test_acc)


/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Model: "model_9"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_10 (InputLayer)       [(None, 32, 32, 3)]       0         
                                                                 
 block1_conv1 (Conv2D)       (None, 32, 32, 64)        1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 32, 32, 64)        36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 16, 16, 64)        0         
                                                                 
 block2_conv1 (Conv2D)       (None, 16, 16, 128)       73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 16, 16, 128)       147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 8, 8, 128)         0   

From the code results above, we get an improvement to 50%. So I will modify the parameters further to see if there will be an improvement


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.preprocessing import OneHotEncoder

# Load the CIFAR 100 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()

# Preprocess the images
x_train_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_train.astype('float32'))
x_test_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_test.astype('float32'))

# One-hot encode the labels
encoder = OneHotEncoder(sparse=False)
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

# Load the pre-trained VGG16 model with the convolutional base
vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Freeze layers up to a certain point
for layer in vgg_base.layers[:-4]:
    layer.trainable = False

# Add Dense layers on top of the convolutional base
x = Flatten()(vgg_base.output)
x = Dense(512, activation='relu')(x)
x = Dense(256, activation='relu')(x)
predictions = Dense(100, activation='softmax')(x)

# Create the final model
model = Model(inputs=vgg_base.input, outputs=predictions)

# Compile the model with RMSprop optimizer and a lower learning rate
rmsprop_optimizer = RMSprop(learning_rate=0.0001)
model.compile(optimizer=rmsprop_optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Print a summary of the model
model.summary()

# Data augmentation
datagen = ImageDataGenerator(
    rotation_range=15,             # Reduce rotation range further
    width_shift_range=0.1,         # Keep width shift range same
    height_shift_range=0.1,        # Keep height shift range same
    shear_range=0.1,               # Keep shear range same
    zoom_range=0.1,                # Keep zoom range same
    horizontal_flip=True,
    fill_mode='nearest'
)

# Train the model with data augmentation
history = model.fit(datagen.flow(x_train_preprocessed, y_train_encoded, batch_size=128),
                    steps_per_epoch=len(x_train_preprocessed) / 128,
                    epochs=10,
                    validation_data=(x_test_preprocessed, y_test_encoded))

# Evaluate the model
test_loss, test_acc = model.evaluate(x_test_preprocessed, y_test_encoded)
print('Test accuracy:', test_acc)



/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Model: "model_10"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_11 (InputLayer)       [(None, 32, 32, 3)]       0         
                                                                 
 block1_conv1 (Conv2D)       (None, 32, 32, 64)        1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 32, 32, 64)        36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 16, 16, 64)        0         
                                                                 
 block2_conv1 (Conv2D)       (None, 16, 16, 128)       73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 16, 16, 128)       147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 8, 8, 128)         0  

There has been a 1% improvement, so I will adjusting the parameters one more time

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.preprocessing import OneHotEncoder

# Load the CIFAR 100 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()

# Preprocess the images
x_train_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_train.astype('float32'))
x_test_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_test.astype('float32'))

# One-hot encode the labels
encoder = OneHotEncoder(sparse=False)
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

# Load the pre-trained VGG16 model with the convolutional base
vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Freeze layers up to a certain point
for layer in vgg_base.layers[:-4]:
    layer.trainable = False

# Add Dense layers on top of the convolutional base
x = Flatten()(vgg_base.output)
x = Dense(512, activation='relu')(x)
x = Dense(256, activation='relu')(x)
predictions = Dense(100, activation='softmax')(x)

# Create the final model
model = Model(inputs=vgg_base.input, outputs=predictions)

# Compile the model with RMSprop optimizer and a lower learning rate
rmsprop_optimizer = RMSprop(learning_rate=0.0001)
model.compile(optimizer=rmsprop_optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Print a summary of the model
model.summary()

# Data augmentation
datagen = ImageDataGenerator(
    rotation_range=25,             # Increase rotation range
    width_shift_range=0.1,         # Keep width shift range same
    height_shift_range=0.1,        # Keep height shift range same
    shear_range=0.1,               # Keep shear range same
    zoom_range=0.1,                # Keep zoom range same
    horizontal_flip=True,
    fill_mode='nearest'
)

# Train the model with data augmentation
history = model.fit(datagen.flow(x_train_preprocessed, y_train_encoded, batch_size=128),
                    steps_per_epoch=len(x_train_preprocessed) / 128,
                    epochs=10,
                    validation_data=(x_test_preprocessed, y_test_encoded))

# Evaluate the model
test_loss, test_acc = model.evaluate(x_test_preprocessed, y_test_encoded)
print('Test accuracy:', test_acc)


/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Model: "model_11"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_12 (InputLayer)       [(None, 32, 32, 3)]       0         
                                                                 
 block1_conv1 (Conv2D)       (None, 32, 32, 64)        1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 32, 32, 64)        36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 16, 16, 64)        0         
                                                                 
 block2_conv1 (Conv2D)       (None, 16, 16, 128)       73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 16, 16, 128)       147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 8, 8, 128)         0  

After performing the data augmentation and adjusting some parameters, we see an improvement. The accuracy is now 51.2 %. Hence, it does make sense to use data augmentation for this model.
 Here's why:
 Improvement in Performance: As evidenced by the comparison between the fine-tuned model without data augmentation and the model with data augmentation, we observe an improvement in accuracy when data augmentation is applied. This indicates that data augmentation helps the model to learn more robust features and generalize better to unseen data.

Mitigation of Overfitting: Data augmentation introduces variability into the training data by applying various transformations such as rotation, shifting, and flipping. This helps to prevent the model from overfitting to the training data and improves its ability to generalize to new, unseen examples.

Enhanced Generalization: By exposing the model to a wider range of variations of the input data during training, data augmentation encourages the model to learn features that are invariant to such variations. As a result, the model becomes more adept at recognizing patterns in the data, leading to improved generalization performance.

Utilization of Available Data: In many cases, the amount of available training data may be limited. Data augmentation allows us to effectively increase the size of the training dataset by generating additional synthetic examples, which can lead to better utilization of the available data and improved model performance.

Overall, data augmentation is a valuable technique for improving the performance of deep learning models, especially when the dataset is limited in size or when there is a risk of overfitting. In the case of fine-tuning a pre-trained model like VGG16, data augmentation can further enhance the model's ability to learn meaningful representations and achieve higher accuracy.

## Question 3
In order to train the full VGG16 network, we'll unfreeze all layers of the VGG16 model and train it with a lower learning rate and potentially a larger batch size.
In the code below,
- The CIFAR-100 dataset is loaded and the images preprocessed.
All layers of the VGG16 model are made trainable by setting layer.trainable = True for each layer.
- Dense layers are added on top of the VGG16 model to perform classification.
- the model is compiled with a RMSprop optimizer with a lower learning rate.
-The model is trained for 10 epochs with a batch size of 256.
- The model is evaluated on the test data

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.preprocessing import OneHotEncoder

# Load the CIFAR 100 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()

# Preprocess the images
x_train_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_train.astype('float32'))
x_test_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_test.astype('float32'))

# One-hot encode the labels
encoder = OneHotEncoder(sparse=False)
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

# Load the pre-trained VGG16 model
vgg_model = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Make all layers trainable
for layer in vgg_model.layers:
    layer.trainable = True

# Add Dense layers on top
x = Flatten()(vgg_model.output)
x = Dense(512, activation='relu')(x)
x = Dense(256, activation='relu')(x)
predictions = Dense(100, activation='softmax')(x)

# Create the final model
model = Model(inputs=vgg_model.input, outputs=predictions)

# Compile the model with RMSprop optimizer and a lower learning rate
rmsprop_optimizer = RMSprop(learning_rate=0.0001)
model.compile(optimizer=rmsprop_optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Print a summary of the model
model.summary()

# Train the model
history = model.fit(x_train_preprocessed, y_train_encoded, epochs=10, batch_size=256, validation_data=(x_test_preprocessed, y_test_encoded))

# Evaluate the model
test_loss, test_acc = model.evaluate(x_test_preprocessed, y_test_encoded)
print('Test accuracy:', test_acc)


/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Model: "model_11"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_12 (InputLayer)       [(None, 32, 32, 3)]       0         
                                                                 
 block1_conv1 (Conv2D)       (None, 32, 32, 64)        1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 32, 32, 64)        36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 16, 16, 64)        0         
                                                                 
 block2_conv1 (Conv2D)       (None, 16, 16, 128)       73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 16, 16, 128)       147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 8, 8, 128)         0  

After training all layers of the network, we get an increase in the accuracy of about 55.8%
We will adjust some parameters and observe the accuracy.
In this modified code:

We've increased the number of epochs to 20 to allow the model to train for longer.
We've reduced the batch size to 128 to potentially introduce more variability during training.
We've kept the learning rate the same at 0.0001.


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.preprocessing import OneHotEncoder

# Load the CIFAR 100 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()

# Preprocess the images
x_train_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_train.astype('float32'))
x_test_preprocessed = tf.keras.applications.vgg16.preprocess_input(x_test.astype('float32'))

# One-hot encode the labels
encoder = OneHotEncoder(sparse=False)
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

# Load the pre-trained VGG16 model
vgg_model = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Make all layers trainable
for layer in vgg_model.layers:
    layer.trainable = True

# Add Dense layers on top
x = Flatten()(vgg_model.output)
x = Dense(512, activation='relu')(x)
x = Dense(256, activation='relu')(x)
predictions = Dense(100, activation='softmax')(x)

# Create the final model
model = Model(inputs=vgg_model.input, outputs=predictions)

# Compile the model with RMSprop optimizer and a lower learning rate
rmsprop_optimizer = RMSprop(learning_rate=0.0001)
model.compile(optimizer=rmsprop_optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Print a summary of the model
model.summary()

# Train the model with adjusted parameters
history = model.fit(x_train_preprocessed, y_train_encoded, epochs=20, batch_size=128, validation_data=(x_test_preprocessed, y_test_encoded))

# Evaluate the model
test_loss, test_acc = model.evaluate(x_test_preprocessed, y_test_encoded)
print('Test accuracy:', test_acc)


/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Model: "model_12"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_13 (InputLayer)       [(None, 32, 32, 3)]       0         
                                                                 
 block1_conv1 (Conv2D)       (None, 32, 32, 64)        1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 32, 32, 64)        36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 16, 16, 64)        0         
                                                                 
 block2_conv1 (Conv2D)       (None, 16, 16, 128)       73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 16, 16, 128)       147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 8, 8, 128)         0  

The test accuracy gotten after training the full network is 59.6%.

This shows that training all layers of the full VGG16 network allows for increased model capacity, fine-tuning of features at multiple levels, task-specific adaptation, and end-to-end representation learning, all of which contribute to the improved accuracy.

## Part 1.2
Inorder to build my own covnet from scratch,  
I will start with data preparation: Prepare the dataset for training and testing the CNN. This could involve loading the images, resizing them to a fixed size (e.g., 32x32 pixels), one hot encoding and splitting the dataset into training and testing sets.



In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# Load the CIFAR 100 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()

# Normalize the pixel values to a range of [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# One-hot encode the labels
encoder = OneHotEncoder(sparse=False)
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

# Split the dataset into training and validation sets
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train_encoded, test_size=0.2, random_state=42)

# Data augmentation
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.experimental.preprocessing.RandomFlip("horizontal_and_vertical"),
  tf.keras.layers.experimental.preprocessing.RandomRotation(0.1),
  tf.keras.layers.experimental.preprocessing.RandomZoom(0.1),
])

# Apply data augmentation to the training data
x_train = data_augmentation(x_train)


/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


## First basic Model

I will define a basic convolutional neural network design for the initial iteration and I will use a fully connected layer for classification, pooling layers for downsampling, and convolutional layers. Additionally, in order to avoid overfitting, we will use fundamental optimization strategies such dropout regularization.

We can begin with the following fundamental architecture:

Input Layer: Three-channel (RGB) input images of 32 by 32 pixels.
Convolutional Blocks: Feature extraction consisting of convolutional layers and max pooling layers.
Convolutional layer output should be flattened before feeding it into fully connected layers.
Fully Connected Layers: Dense layers used in categorization.
Multiclass classification using Softmax activation is the output layer.


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Define the model architecture
model = Sequential([
    # Convolutional blocks
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),

    # Flatten layer
    Flatten(),

    # Fully connected layers
    Dense(512, activation='relu'),
    Dropout(0.5),  # Dropout regularization to prevent overfitting
    Dense(256, activation='relu'),
    Dropout(0.5),

    # Output layer
    Dense(100, activation='softmax')  # 100 classes for CIFAR-100 dataset
])

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Print model summary
model.summary()


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 32, 32, 32)        896       
                                                                 
 max_pooling2d (MaxPooling2  (None, 16, 16, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 16, 16, 64)        18496     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 8, 8, 64)          0         
 g2D)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 8, 8, 128)         73856     
                                                                 
 max_pooling2d_2 (MaxPoolin  (None, 4, 4, 128)        

### Model Training
The model will be trained on the CIFAR-100 dataset and the performance evaluated on the validation set. early stopping is used to prevent overfitting and monitor the training process.

The initial result gotten gives an accuracy of 23%

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Define the CNN model architecture
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(100, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Define early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train the model
history = model.fit(x_train, y_train, epochs=20, batch_size=32, validation_data=(x_val, y_val), callbacks=[early_stopping])

# Evaluate the model on the validation set
val_loss, val_accuracy = model.evaluate(x_val, y_val)
print("Validation Loss:", val_loss)
print("Validation Accuracy:", val_accuracy)


Epoch 1/20
1250/1250 [==============================] - 11s 6ms/step - loss: 4.3516 - accuracy: 0.0312 - val_loss: 4.1506 - val_accuracy: 0.0545
Epoch 2/20
1250/1250 [==============================] - 7s 5ms/step - loss: 4.0453 - accuracy: 0.0732 - val_loss: 3.8045 - val_accuracy: 0.1043
Epoch 3/20
1250/1250 [==============================] - 7s 5ms/step - loss: 3.8501 - accuracy: 0.1021 - val_loss: 3.6281 - val_accuracy: 0.1394
Epoch 4/20
1250/1250 [==============================] - 7s 6ms/step - loss: 3.7068 - accuracy: 0.1243 - val_loss: 3.4510 - val_accuracy: 0.1727
Epoch 5/20
1250/1250 [==============================] - 7s 5ms/step - loss: 3.6094 - accuracy: 0.1407 - val_loss: 3.3746 - val_accuracy: 0.1802
Epoch 6/20
1250/1250 [==============================] - 7s 5ms/step - loss: 3.5249 - accuracy: 0.1558 - val_loss: 3.3036 - val_accuracy: 0.1985
Epoch 7/20
1250/1250 [==============================] - 7s 5ms/step - loss: 3.4501 - accuracy: 0.1699 - val_loss: 3.3374 - val_accuracy

Inorder to improve the model accuracy, I will try incorporating batch normalization, increasing dropout regularization, and implementing a learning rate schedule as follows:
- batch normalization layers are added after each convolutional layer to stabilize and speed up training.
- An increase in the dropout regularization rate to 0.5 after each dense layer to reduce overfitting.
- inclusion of two callbacks: early stopping to monitor the validation loss and reduce learning rate on plateau to adjust the learning rate dynamically during training.

The results show an improvement to about 29% in accuracy

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Define the CNN model architecture with advanced features
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(100, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Define early stopping and learning rate schedule callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6)

# Train the model
history = model.fit(x_train, y_train, epochs=20, batch_size=32, validation_data=(x_val, y_val), callbacks=[early_stopping, reduce_lr])

# Evaluate the model on the validation set
val_loss, val_accuracy = model.evaluate(x_val, y_val)
print("Validation Loss:", val_loss)
print("Validation Accuracy:", val_accuracy)


Epoch 1/20
1250/1250 [==============================] - 12s 7ms/step - loss: 4.3938 - accuracy: 0.0439 - val_loss: 3.9796 - val_accuracy: 0.0943 - lr: 0.0010
Epoch 2/20
1250/1250 [==============================] - 8s 6ms/step - loss: 4.0559 - accuracy: 0.0770 - val_loss: 3.7921 - val_accuracy: 0.1183 - lr: 0.0010
Epoch 3/20
1250/1250 [==============================] - 7s 6ms/step - loss: 3.8810 - accuracy: 0.1043 - val_loss: 3.6580 - val_accuracy: 0.1325 - lr: 0.0010
Epoch 4/20
1250/1250 [==============================] - 8s 7ms/step - loss: 3.7427 - accuracy: 0.1218 - val_loss: 3.5394 - val_accuracy: 0.1597 - lr: 0.0010
Epoch 5/20
1250/1250 [==============================] - 9s 7ms/step - loss: 3.6340 - accuracy: 0.1416 - val_loss: 3.4791 - val_accuracy: 0.1658 - lr: 0.0010
Epoch 6/20
1250/1250 [==============================] - 10s 8ms/step - loss: 3.5382 - accuracy: 0.1544 - val_loss: 3.2401 - val_accuracy: 0.2117 - lr: 0.0010
Epoch 7/20
1250/1250 [==============================] - 

### Further improvement in Model

Inorder to improve performance I will try the following:

Increase Model Depth: We can add more convolutional layers to increase the depth of the model. Deeper models have more capacity to learn complex patterns but may also be prone to overfitting.

Adjust Layer Sizes: We can experiment with increasing the number of filters in each convolutional layer or increasing the number of neurons in the dense layers to give the model more capacity to learn.

Residual Connections: We can add residual connections, which allow gradients to flow more easily during training and can help alleviate the vanishing gradient problem in deeper models.

Model Ensembling: We can train multiple models with different architectures or random initializations and combine their predictions to improve accuracy.

From the results, there is an improvement in accuracy to about 32%

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define the CNN model architecture with increased depth and additional convolutional layers
model = Sequential([
    Conv2D(64, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    BatchNormalization(),
    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Conv2D(256, (3, 3), activation='relu'),
    BatchNormalization(),
    Conv2D(256, (3, 3), activation='relu'),
    BatchNormalization(),
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(100, activation='softmax')
])

# Compile the model with Adam optimizer and learning rate scheduling
optimizer = Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Define early stopping and learning rate schedule callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6)

# Train the model with data augmentation
history = model.fit(datagen.flow(x_train_normalized, y_train, batch_size=32),
                    epochs=20,
                    steps_per_epoch=len(x_train_normalized) / 32,
                    validation_data=(x_val_normalized, y_val),
                    callbacks=[early_stopping, reduce_lr])

# Evaluate the model on the validation set
val_loss, val_accuracy = model.evaluate(x_val_normalized, y_val)
print("Validation Loss:", val_loss)
print("Validation Accuracy:", val_accuracy)


Epoch 1/20
1250/1250 [==============================] - 42s 25ms/step - loss: 4.4177 - accuracy: 0.0387 - val_loss: 4.0565 - val_accuracy: 0.0705 - lr: 0.0010
Epoch 2/20
1250/1250 [==============================] - 40s 32ms/step - loss: 4.1584 - accuracy: 0.0613 - val_loss: 3.8692 - val_accuracy: 0.0969 - lr: 0.0010
Epoch 3/20
1250/1250 [==============================] - 41s 32ms/step - loss: 4.0115 - accuracy: 0.0791 - val_loss: 3.7065 - val_accuracy: 0.1165 - lr: 0.0010
Epoch 4/20
1250/1250 [==============================] - 30s 24ms/step - loss: 3.8960 - accuracy: 0.0943 - val_loss: 3.6394 - val_accuracy: 0.1335 - lr: 0.0010
Epoch 5/20
1250/1250 [==============================] - 29s 23ms/step - loss: 3.7804 - accuracy: 0.1125 - val_loss: 3.5209 - val_accuracy: 0.1441 - lr: 0.0010
Epoch 6/20
1250/1250 [==============================] - 31s 25ms/step - loss: 3.6809 - accuracy: 0.1291 - val_loss: 3.5013 - val_accuracy: 0.1606 - lr: 0.0010
Epoch 7/20
1250/1250 [========================

### Model Simplification
Reducing the model complexity is an approach to address potential overfitting or training difficulties.
Here, the model architecture is simplified by reducing the number of layers and parameters in an attempt to improve the accuracy.
In this simplified model:

-the number of convolutional layers is reduced to two and decreased the number of filters in each layer to 32 and 64, respectively.
- Removal of one dense layer and reduced the number of neurons in the remaining dense layer to 128.

The results from the simple model are lower with an accuracy of 25%. Hence a rather sophisticated model is better in this case.


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Define the CNN model architecture with reduced complexity
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(100, activation='softmax')
])

# Compile the model with Adam optimizer
optimizer = Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Define early stopping and learning rate schedule callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6)

# Train the simplified model
history = model.fit(x_train_normalized, y_train, batch_size=32, epochs=20, validation_data=(x_val_normalized, y_val), callbacks=[early_stopping, reduce_lr])

# Evaluate the model on the validation set
val_loss, val_accuracy = model.evaluate(x_val_normalized, y_val)
print("Validation Loss:", val_loss)
print("Validation Accuracy:", val_accuracy)


Epoch 1/20
1250/1250 [==============================] - 8s 4ms/step - loss: 4.1669 - accuracy: 0.0656 - val_loss: 3.7326 - val_accuracy: 0.1373 - lr: 0.0010
Epoch 2/20
1250/1250 [==============================] - 7s 5ms/step - loss: 3.7744 - accuracy: 0.1185 - val_loss: 3.4782 - val_accuracy: 0.1703 - lr: 0.0010
Epoch 3/20
1250/1250 [==============================] - 5s 4ms/step - loss: 3.5771 - accuracy: 0.1488 - val_loss: 3.3229 - val_accuracy: 0.1971 - lr: 0.0010
Epoch 4/20
1250/1250 [==============================] - 6s 5ms/step - loss: 3.4587 - accuracy: 0.1682 - val_loss: 3.2891 - val_accuracy: 0.2003 - lr: 0.0010
Epoch 5/20
1250/1250 [==============================] - 6s 5ms/step - loss: 3.3731 - accuracy: 0.1806 - val_loss: 3.1915 - val_accuracy: 0.2229 - lr: 0.0010
Epoch 6/20
1250/1250 [==============================] - 9s 7ms/step - loss: 3.3114 - accuracy: 0.1913 - val_loss: 3.1839 - val_accuracy: 0.2200 - lr: 0.0010
Epoch 7/20
1250/1250 [==============================] - 6s

The final model that performed best had an accuracy of 32.4 % .


The model architecture consists of multiple convolutional layers followed by batch normalization, max-pooling layers, and dense layers. The convolutional layers extract features from the input images, while batch normalization helps stabilize and speed up training. Max-pooling layers downsample the feature maps to reduce computation, and dense layers perform classification.
- The Adam optimizer with a learning rate of 0.001 is used
- Callbacks: Early stopping and learning rate reduction callbacks are employed
- Data Augmentation: The model is trained with data augmentation using the ImageDataGenerator class to generate augmented images in real-time during training.

